In [ ]:
import mne
import numpy as np
import plotly.express as px
from crosci.biomarkers import (
    DFA,
    fEI
)

In [ ]:
processed = mne.io.read_raw_fif('/home/jackhoyle/mne_data/ds003969/processed_v4/sub-001_task-med2_v4_processed_raw.fif', preload=True)

In [ ]:
def compute_dfa_fei(
    processed: mne.io.BaseRaw, 
    bandpass_low: float,
    bandpass_high: float,
    epoch_len: float = 180.0, 
    dfa_min_window: float = 5., 
    dfa_max_window: float = 30.,
    fei_window_size: float = 5.,
    fei_window_overlap: float = 0.8
    ) -> tuple[np.ndarray, np.ndarray]:
    epochs = mne.make_fixed_length_epochs(
        processed.pick('eeg'), duration=epoch_len, preload=True, reject_by_annotation=True, verbose='error'
    )
    epochs.filter(l_freq=bandpass_low, h_freq=bandpass_high, verbose=False)
    epochs.apply_hilbert(envelope=True)

    dfa_data = [] 
    fEI_data = [] 

    for epoch in epochs:
        # DFA
        (DFA_array, window_sizes, fluctuations, DFA_intercept) = DFA(
            epoch,
            processed.info['sfreq'],
            [dfa_min_window, dfa_max_window],
            [dfa_min_window, dfa_max_window],
            overlap=True
        )
        dfa_data.append(DFA_array[:, np.newaxis])

        # fEI
        (fEI_outliers_removed, fEI_val, num_outliers, wAmp, wDNF) = fEI(
                epoch,
                processed.info['sfreq'],
                fei_window_size,
                fei_window_overlap,
                DFA_array
            )
        fEI_data.append(fEI_outliers_removed)

    # averages by channel over epochs
    dfa_avg = np.nanmean(np.concatenate(dfa_data, axis=1), axis=1)
    fEI_avg = np.nanmean(np.concatenate(fEI_data, axis=1), axis=1)

    return dfa_avg, fEI_avg

In [ ]:
freqbands = {
    'delta': (1.0, 4.0),
    'theta': (4.0, 8.0),
    'alpha': (8.0, 13.0),
    'beta': (13.0, 30.0),
    'broadband': (1.0, 30.0)
}

In [ ]:
dfas = dict()
feis = dict()
for freq_name, (bandpass_low, bandpass_high) in freqbands.items():
    dfa_avg, fEI_avg = compute_dfa_fei(
        processed=processed,
        bandpass_low=bandpass_low,
        bandpass_high=bandpass_high
    )
    dfas[freq_name] = dfa_avg
    feis[freq_name] = fEI_avg
    print(f"{freq_name}: DFA avg = {np.nanmean(dfa_avg)}, fEI avg = {np.nanmean(fEI_avg)}")